# Act 1 — High availability & disaster recovery

Every production workload eventually has to answer two questions: *can we survive a failure?* and *how fast can we come back?* High availability is the design for surviving the everyday failures — an instance dies, an Availability Zone wobbles, a deploy goes bad. Disaster recovery is the design for surviving the rare-but-catastrophic ones — a whole region is unreachable, ransomware encrypts the primary database, a faulty change deletes a production bucket.

The two share machinery — replication, backups, failover orchestration — but they answer different SLAs. Treat them as separate budgets, not as one bucket of "resilience work."

## RTO and RPO — the only two numbers that matter

The entire conversation about HA and DR reduces to two numbers, *per workload*:

- **RTO (Recovery Time Objective)** — how long the workload can be down before the business is in real trouble.
- **RPO (Recovery Point Objective)** — how much data the business can afford to lose, measured in time (the last successful backup or replication point).

Quote them per workload, not per company. The customer-facing checkout flow might need RTO under 5 minutes and RPO under 30 seconds. The back-office reporting warehouse might be fine with RTO 24 hours and RPO 4 hours. Every architectural choice that follows — synchronous vs asynchronous replication, multi-AZ vs multi-region, hot standby vs cold backup — is paying for a smaller RTO/RPO. The question is always *how small is small enough for this workload?*

Two costs scale with tightening RTO/RPO: **money** (more hardware, more replicas, more cross-region bandwidth) and **complexity** (more failover automation, more drift between primary and standby, more things to test). Buying multi-region active-active for everything is the costliest mistake in DR planning.

## HA patterns within a Region

Within a Region, AWS gives you three Availability Zones — physically separate data centres with independent power and network — and HA design is mostly about using them.

The standard pattern: **stateless tier across multiple AZs behind a load balancer, stateful tier replicated across AZs, autoscaling on demand**. EC2 in an Auto Scaling Group spanning three AZs behind an ALB. RDS Multi-AZ with a synchronous standby in another AZ. ElastiCache Redis with Multi-AZ replication. S3 — the only AWS service where you don't think about AZs because it stores objects across them automatically.

Four AWS-native HA primitives worth being explicit about:

- **Multi-AZ Auto Scaling Group** — kills the *single host* failure mode at zero developer cost. Set `min` ≥ 2, span all AZs, attach a target group, enable cross-zone load balancing.
- **RDS Multi-AZ (synchronous)** — fails over the writer endpoint to the standby in another AZ on hardware or maintenance failure. RPO ≈ 0, RTO ≈ 60–120 seconds. Single-region only.
- **Aurora's storage layer** — six copies of every write across three AZs at the storage tier itself. You can lose two copies (an AZ) and the database keeps running. Replicas in other AZs are read-only and can be promoted in seconds.
- **DynamoDB** — multi-AZ replication is invisible to you. Tables are durable across the Region by design.

The failure mode HA *doesn't* handle is a whole-region outage. For that, you move to DR.

## DR strategies — the four AWS shapes

AWS frames DR as four named strategies, ordered by RTO/RPO (and cost). Pick the one that matches each workload's targets — many enterprises run two or three different strategies across their portfolio.

| Strategy | RPO | RTO | Idea |
|---|---|---|---|
| **Backup & Restore** | hours | hours–days | Periodic backups to S3 / AWS Backup. On disaster, provision new infra in DR Region and restore from backup. Cheapest, slowest. |
| **Pilot Light** | minutes | tens of minutes | Core data continuously replicated to DR Region; minimal compute kept off or scaled down. On disaster, scale up and switch DNS. |
| **Warm Standby** | seconds–minutes | minutes | Full but scaled-down stack always running in DR Region, kept current via replication. On disaster, scale up and switch traffic. |
| **Multi-Site Active-Active** | seconds | seconds | Both Regions serve traffic; data replicated bidirectionally or via a globally consistent store. On disaster, traffic shifts automatically. Costliest, fastest. |

The progression is also a progression in *how much of production you keep warm in the DR Region* — none, just the data, the data plus a small fleet, or a full second production. Aurora Global Database, DynamoDB Global Tables, S3 Cross-Region Replication, and Route 53 health-check failover are the building blocks under every option above Backup & Restore.

The non-negotiable across all four: **test the failover**. A DR plan you have never run is a plan, but it is not a tested plan, and the difference is everything when the disaster is real.

## AWS Backup

**AWS Backup** is the managed, central backup service that covers EC2, EBS, RDS, Aurora, DynamoDB, EFS, FSx, S3, Storage Gateway, Neptune, DocumentDB, and Redshift — the long list is the point. Without it, you would run six different backup configurations per service.

The model has three pieces.

- **Backup vaults** hold the recovery points. Each vault has a KMS key for encryption and an access policy. **Vault Lock** makes the vault WORM — once locked in compliance mode, even the root user cannot shorten retention or delete recovery points. The ransomware-resilience answer.
- **Backup plans** define schedules, retention, and lifecycle transitions to cold storage. One plan can target many resource types.
- **Resource assignment** matches resources to plans by tag (or explicit ARN). Tag `backup=daily-prod` on the resources you want covered; the plan picks them up automatically.

Two features worth highlighting. **Cross-Region copy** as part of the backup plan automatically duplicates recovery points to another Region — the cheapest DR option for many workloads. **Cross-account copy** uses Backup Vaults in a separate AWS account, isolating backups from a compromise of the primary account.

AWS Backup is *backup*, not *replication*. RPO is the schedule interval (typically hours). For RPO in seconds, you need engine-level replication — Aurora Global Database, DynamoDB Global Tables, S3 CRR — not Backup.

# Act 2 — Cost levers

Cloud cost is not magic and it is not a fixed line item. It is the cumulative outcome of dozens of choices — instance sizes, purchase commitments, storage tiers, idle resources, transfer paths. The teams that pay 30% of list for the same workload as another team are not getting a secret discount; they are pulling more of the levers, more deliberately.

This act walks the levers that move the bill the most, in roughly the order of impact, plus the tools that show you where to pull.

## Right-sizing and idle elimination

The first lever on every cost review: **resources that aren't doing the work they're sized for**. Two patterns:

- **Over-sized** — an m5.4xlarge running at 8% CPU for the last 90 days. Drop to m5.large or move to a Graviton-based equivalent and save 60%+.
- **Idle** — an EC2 instance that gets no requests but was forgotten after a project ended. A dev RDS that runs 24/7 but is only used at office hours. An EBS volume detached from any instance for months. An Elastic IP that nobody is using.

**Compute Optimizer** is AWS's right-sizing engine. It analyses 14 days of CloudWatch data for EC2, Auto Scaling Groups, EBS volumes, Lambda functions, ECS services, and RDS instances, and recommends a specific smaller (or differently-shaped) resource. The recommendations are concrete — "switch this instance to m6g.large, projected monthly savings $X" — and account for performance impact.

**Trusted Advisor** flags idle resources in the Cost Optimization category — idle load balancers, underutilised EBS volumes, low-utilisation EC2 instances. Free at the basic tier; the full set requires Business or Enterprise support.

The operational pattern: weekly review of Compute Optimizer + Trusted Advisor + Cost Explorer, with right-sizing PRs the same week. Costs that aren't reviewed grow.

## Commitment discounts — Reserved Instances, Savings Plans, Spot

For anything running steadily, paying On-Demand is leaving 30–70% on the table. AWS offers three families of discount:

- **Reserved Instances (RIs)** — commit to a specific instance family + region for 1 or 3 years; up to ~72% off On-Demand. Two flavours: **Standard RIs** (lowest discount, locked to the exact family) and **Convertible RIs** (smaller discount but can be exchanged for different families during the term). Best for predictable steady-state on a single family.
- **Savings Plans (SPs)** — commit to an hourly *dollar amount* of spend; the discount applies across families and regions automatically. **Compute Savings Plans** cover EC2, Fargate, and Lambda interchangeably. **EC2 Instance Savings Plans** are family-pinned but offer slightly deeper discounts. SPs are easier to manage than RIs for fleets that churn across families.
- **Spot Instances** — bid on unused EC2 capacity, up to ~90% off On-Demand. AWS can reclaim the instance with two minutes' notice. Perfect for batch jobs, CI runners, stateless workers, and Fargate Spot tasks that can be restarted.

A well-run fleet often pays 70–85% off list: SPs on the steady-state base, Spot on the burst layer, Compute Optimizer keeping the base sized right. RIs make sense for stable workloads pinned to a single family — RDS reserved instances in particular.

**S3 Intelligent-Tiering** is the storage equivalent: S3 moves objects between Frequent / Infrequent / Archive Instant / Archive / Deep Archive automatically based on access patterns. For data with unpredictable access — log archives, mixed-workload data lakes — it consistently beats hand-managed lifecycle policies.

## Cost tools — Cost Explorer, Budgets, Anomaly Detection

Three tools, three jobs.

**Cost Explorer** is the interactive analysis console. Slice spend by service, account, region, tag, or any combination, over arbitrary time windows. The recommendations tab proposes RIs and Savings Plans based on the last 7/30/60 days of usage. The forecast view projects month-end and next-month spend from the trend. Cost Explorer is where finance and engineering meet — same data, two questions.

**AWS Budgets** are the proactive alarms. Set a monthly spend budget, a usage budget (e.g. "reserved instance coverage > 80%"), or a Savings Plans utilisation budget, and Budgets emails or fires SNS when you cross thresholds (50%, 80%, 100%, forecasted overage). The trick is to budget *per team or per workload*, not just for the account total — a one-team overrun is invisible inside a healthy whole-account budget.

**AWS Cost Anomaly Detection** is the ML layer on top: it learns each service's normal spending pattern and alerts on unexpected spikes. The classic catch is the cryptominer that compromised an IAM key — an Anomaly Detector flags the EC2 spend explosion before the bill arrives. Subscribe at the service or linked-account level; route alerts to SNS, Chatbot, or email.

**Cost and Usage Reports (CUR)** are the deep export — hourly resource-level CSV/parquet to S3, queryable from Athena or QuickSight. The chargeback substrate for any organisation big enough to need it.

# Act 3 — Migration

Most AWS customers do not arrive in greenfield. They arrive with a data centre, a portfolio of workloads, and a deadline. **Migration** is the discipline of moving from there to here without breaking the business in transit — and AWS has shaped a product family around it.

The framing AWS pushes is the **7 Rs**, popularised by Gartner: every workload is being **Retired**, **Retained**, **Rehosted**, **Relocated**, **Replatformed**, **Repurchased**, or **Refactored**. Picking the R is half the migration; the other half is the service that moves the bytes.

## The 7 Rs

| R | What it means | When |
|---|---|---|
| **Retire** | Turn it off | The workload is unused or duplicated |
| **Retain** | Leave on-prem | Latency, compliance, or vendor lock makes the move uneconomic |
| **Rehost** | Lift-and-shift to EC2 / VMware Cloud on AWS | Fastest move; minimal change. *"Forklift to the cloud."* |
| **Relocate** | Move VMware workloads to VMware Cloud on AWS | Specifically VMware → VMware on AWS |
| **Replatform** | Lift-and-tinker — small optimisations on the way (e.g. database to RDS) | Modest gains without app rewrite |
| **Repurchase** | Replace with SaaS — drop the licensed CRM, buy Salesforce | When a SaaS equivalent exists |
| **Refactor** | Rebuild for the cloud — break monolith into microservices, go serverless | Highest value, highest effort, longest timeline |

The practical sequencing most enterprises follow: **Retire and Retain first** (cut scope), then **Rehost the majority** to get out of the data centre fast, then **Replatform and Refactor over time** as workloads earn the investment. The two-phase pattern — "get there, then make it cloud-native" — almost always beats the "refactor everything in flight" temptation.

## Migration services

**AWS Application Migration Service (MGN)** — formerly CloudEndure Migration — is the modern lift-and-shift tool. Install a lightweight agent on each source server (physical, VMware, Hyper-V, cross-cloud), and MGN continuously block-level replicates the disks to a low-cost staging area on EBS. When you're ready, you launch test instances or cut over — the source machine becomes an EC2 instance in minutes, byte-identical. Replaces the older AWS Server Migration Service (SMS), which is deprecated.

**AWS Database Migration Service (DMS)** handles the database tier. Two main modes: **homogeneous** (Oracle → RDS Oracle, SQL Server → RDS SQL Server — schema is the same) and **heterogeneous** (Oracle → Aurora PostgreSQL, SQL Server → Aurora MySQL). Heterogeneous migrations use the **Schema Conversion Tool (SCT)** to translate schema and stored procedures, then DMS streams the data with ongoing CDC so the source and target stay in sync until cutover. Near-zero-downtime database moves.

**AWS DataSync** is the network-based bulk file transfer service. Used for moving large NFS/SMB/S3/EFS/HDFS datasets — TB to PB — over the network with built-in encryption, integrity checks, and bandwidth controls. Faster than the source-system tools (`rsync`, `aws s3 sync`) because the agent is purpose-built and parallelises aggressively.

**AWS Transfer Family** is managed SFTP / FTPS / FTP / AS2 for ongoing file exchange with partners and customers. Different problem than migration (it's a service for steady-state file exchange, not a one-shot move), but it shows up when the migration story involves "we need to keep accepting files from this vendor over SFTP after we leave the data centre."

**AWS Migration Hub** is the umbrella dashboard — a single place to track migrations across MGN, DMS, and partner tools, with portfolio-level reporting and the Migration Hub Strategy Recommendations service that proposes Rs based on application discovery data.

## Snow family — bulk data when the network won't do

For truly large datasets, the network is too slow. The classic example: 500 TB of imagery on a 100 Mbps WAN link would take roughly 460 days to upload. AWS will ship you a physical device instead.

- **AWS Snowcone** — a small ruggedised edge device, 8 TB usable, battery-powered, perfect for IoT / military / field collection. Includes some on-device compute.
- **AWS Snowball Edge** — the workhorse. Two variants: **Storage Optimized** (up to ~80 TB usable) and **Compute Optimized** (more vCPU + GPU, less storage, for edge compute). Ships in a hardened case.
- **AWS Snowmobile** — a literal 45-foot shipping container delivering 100 PB per Snowmobile. For exabyte-scale data-centre evacuations.

The workflow is the same across the smaller devices: order from the console, AWS ships the device, you load data over local NFS/SMB/S3-compatible API, ship it back, AWS ingests into S3. Encryption is end-to-end with KMS keys you control.

Rule of thumb: if shipping a Snowball is faster than your network link can sustain over the migration window, ship the box. The math usually favours physical transport above tens of terabytes on most corporate connections.

## Putting it together

HA, DR, cost, and migration share a discipline: **measure first, design for the actual targets, and rehearse the recovery paths.**

- **HA** — span AZs by default for production tiers, use the AWS-native HA primitives (Multi-AZ ASG, RDS Multi-AZ, Aurora's storage layer), and trust S3 / DynamoDB / managed services to do the AZ work for you.
- **DR** — quote RTO/RPO per workload, pick the matching strategy (Backup & Restore, Pilot Light, Warm Standby, or Active-Active), and run quarterly failover drills. Aurora Global Database, DynamoDB Global Tables, S3 CRR, and Route 53 health-check failover are the building blocks.
- **Cost** — right-size weekly with Compute Optimizer, commit on the steady-state base with Savings Plans, burst on Spot, lifecycle storage with Intelligent-Tiering, and budget per team. Set up Cost Anomaly Detection on day one.
- **Migration** — pick the R per workload, lift-and-shift the bulk with MGN, move databases with DMS (with SCT for heterogeneous), DataSync the file estates, and ship a Snowball for anything beyond the network's reach. Track the portfolio in Migration Hub.

The systems that survive and scale are the ones whose operators rehearse the bad days. The cost of disciplined HA, DR, cost, and migration is small; the cost of skipping any of them is paid all at once, usually in public.